In [42]:
# Report on the presence of top cities in the Dewey data

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
DEWEY_PATH = os.getenv("DEWEY_PATH")

COLUMNS = [
    'JURISDICTION', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'STATUS_NORMALIZED', 'RECORD_TYPE_ORIGINAL', 'APN', 'STREET', 'ZIPCODE'
]  # Columns to load from Dewey data file

DEWEY_LA_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "dewey_ca_la_county_permits.parquet")
LADBS_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data", "Building_and_Safety_-_Building_Permits_Issued_Between_2010_and_2019_(N)_20260721.csv")


In [43]:
# Load the data

dfd = pd.read_parquet(DEWEY_LA_FILEPATH, columns=COLUMNS)
dfl = pd.read_csv(LADBS_FILEPATH)

/var/folders/ln/_cyxjf216xsc39dhb2hsjdz00000gn/T/ipykernel_98254/799795480.py:4: DtypeWarning: Columns (0: CD) have mixed types. Specify dtype option on import or set low_memory=False.
  dfl = pd.read_csv(LADBS_FILEPATH)


In [44]:
# Check counts by year under Dewey for Los Angeles

dfd['PERMIT_YEAR'] = dfd['PERMIT_DATE'].dt.year
dfdy = dfd.loc[
    (dfd['JURISDICTION'] == 'Los Angeles')  & 
    (dfd['RECORD_TYPE_ORIGINAL'] == 'Bldg-New') &
    (dfd['STATUS_NORMALIZED'].isin(['Final', 'Active', 'Inactive'])) # LADBS data does not include in-review permits
].groupby('PERMIT_YEAR').agg(DEWEY_COUNT=('PERMIT_YEAR', 'count')).reset_index()


In [45]:
# Check counts by year under LADBS data

dfl['ISSUE_DATE'] = pd.to_datetime(dfl['ISSUE_DATE'], errors='coerce')
dfl['PERMIT_YEAR'] = dfl['ISSUE_DATE'].dt.year
dfly = dfl.loc[
    (dfl['PERMIT_TYPE'] == 'Bldg-New')
].groupby('PERMIT_YEAR').agg(LADBS_COUNT=('PERMIT_YEAR', 'count')).reset_index()


In [46]:
# Check match

dfy = dfly.merge(dfdy, on='PERMIT_YEAR', how='inner')
dfy

,PERMIT_YEAR,LADBS_COUNT,DEWEY_COUNT
0,2010,1984,1996
1,2011,1791,1796
2,2012,2134,2121
3,2013,2449,2439
4,2014,3213,3213
5,2015,3574,3569
6,2016,3638,3621
7,2017,4285,4265
8,2018,4602,4610
9,2019,4505,4494


In [47]:
# Conclusion:
# - Dewey data is quite accurate for Los Angeles 
# - Dewey data contains electrical work permits that the LADBS data does not include
# - Assume Dewey data counts are accurate for other cities 